# Object Builder Library (Grid Atlas)

## Introduction to Object Builder

This report introduces a new CIM-Builder library named "Object Builder", that enables end user to add new CIM-based equipments to existing test system data in a reliable manner. 

<p>
<!--<img src="Report_Figures/Robot_in_Workshop.png" alt="My Plot" align="right" width="10%"/>-->
The library takes as input the original eXtensible Markup Language (XML) based model along with the specificaiton of the new element to be added. The object builder integrates the requested element into the appropraite location within the network, ensuring consistency with power system operational requirements. 
<img src="Report_Figures/Object_Builder_Overview_Figure.png" alt="My Plot" align="left" style="margin: 20px; width: 40%"/>

The resulting network is then output as a modified XML file. This process allows end users to construct Grid Atlas models in a streamlined, efficient, and CIM-compliant manner.
<!--</p>
<br><br>
<p>-->
A simple overview of the Object Builder, specifying inputsm outputs and major blocks are shown in Figure. 
</p>





## How the Object Builder Works

### Catalog Parser


The two-part catalog parser function is provided below. First let us load a network to use as base test system.

In [3]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
container = feeder
network = FeederModel(container=feeder, connection=file)

This initializes and loads the "IEEE13.xml" file which contains asset and network data of IEEE 13 node network. The main Catalog Parser function is introduced next.

In [2]:
import logging
import json 
from cimgraph.models import GraphModel
from __future__ import annotations


from cimgraph.databases import get_cim_profile
_log = logging.getLogger(__name__)

def catalog_parser(catalog_file, network):
    file = open(catalog_file) ## opening json file
    catalog = json.load(file) ## loading jsonj file contents as catalog
    data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
    cim_profile, cim = get_cim_profile() # Import CIM profile 
    obj = item_parser(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
    file.close()
    return obj

def item_parser(data:dict, network: GraphModel, cim):
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data:
        if type(data[attribute]) == str:
            if attribute in class_type.__dataclass_fields__:
                setattr(obj, attribute, data[attribute])
            else:
                _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list:
            if attribute in class_type.__dataclass_fields__:
                values = getattr(obj, attribute)
                for item in data[attribute]:
                    value = item_parser(item, network, cim)
                    values.append(value)
                setattr(obj, attribute, values)
    return obj



The catalog parser in addition to the original network (passed as input variable "network"), takes in the json catalog file of the new element to be added. The input is converted to a dicitonary form and is fed into the second function named item parser. Item parser is a generic function which can extract the classes and attributes of any CIM objects. Item parser first identifies the main class and corresponding attributes. Then it iterates through each attributes adding them to the network.

Next, let's call the Catalog Parser function and see it working.

In [4]:
Catalog_JSON_file_path = '../test_models/hv69_12.json' ### Power Transformer json file path
Obj_CP_output = catalog_parser(Catalog_JSON_file_path, network) ### Catalog file reads the input and returns an object 

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


The updated network is saved in "Obj_CP_output" variables after running previous code block.

## CIMparator for Verifying Object Builder Functionalities

CIMparator is a Python-based script designed to verify the successful addition of components specified by the user through the Object Builder. XML files, by their nature, are verbose and not easily interpretable by humans. However, they store information in a hierarchical tree structure, a property leveraged by the CIMparator for systematic analysis.

At the core of CIMparator is the CIMparser function, which extracts the hierarchical structure of classes, attributes, and values from an input XML file. The CIMparator tool accepts two XML files—typically the original and the modified network representations. Each file is parsed using the CIMparser function to systematically extract their respective data structures. The two resulting datasets are then compared to identify and highlight the modifications made by the Object Builder library.

In [ ]:
from lxml import etree
from collections import defaultdict
import os
import csv

# === Normalize values for robust comparison ===
def normalize_value(val):
    if val is None:
        return ""
    val = val.strip()

    # Normalize booleans (case-insensitive)
    if val.lower() == "true":
        return True
    if val.lower() == "false":
        return False

    # Normalize numbers
    try:
        return float(val)
    except ValueError:
        return val.lower()  # For case-insensitive string comparison

# === Parse CIM XML into class-instance-attribute dictionary ===
def parse_cim(xml_file):
    ns = {
        'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
        'cim': 'http://iec.ch/TC57/CIM100#'
    }

    tree = etree.parse(xml_file)
    root = tree.getroot()

    class_data = defaultdict(dict)  # {class: {instance_id: {attr: value}}}

    for element in root:
        if not isinstance(element.tag, str):
            continue

        cls = etree.QName(element.tag).localname
        inst_id = element.get('{%s}ID' % ns['rdf']) or element.get('{%s}about' % ns['rdf'])

        attr_map = {}
        for child in element:
            if not isinstance(child.tag, str):
                continue
            attr = etree.QName(child.tag).localname
            attr_value = child.text or child.get('{%s}resource' % ns['rdf'])
            attr_map[attr] = attr_value

        class_data[cls][inst_id] = attr_map

    return class_data

# === Compare two parsed CIM dictionaries ===
def compare_cim_files(data1, data2):
    all_classes = set(data1.keys()) | set(data2.keys())
    differences = []

    for cls in sorted(all_classes):
        ids1 = data1.get(cls, {})
        ids2 = data2.get(cls, {})

        all_ids = set(ids1.keys()) | set(ids2.keys())

        for inst_id in sorted(all_ids):
            if inst_id not in ids1:
                differences.append((cls, inst_id, "ADDED", ids2[inst_id]))
            elif inst_id not in ids2:
                differences.append((cls, inst_id, "REMOVED", ids1[inst_id]))
            else:
                attrs1 = ids1[inst_id]
                attrs2 = ids2[inst_id]

                changes = {}
                all_attrs = set(attrs1.keys()) | set(attrs2.keys())

                for attr in all_attrs:
                    v1 = normalize_value(attrs1.get(attr))
                    v2 = normalize_value(attrs2.get(attr))
                    if v1 != v2:
                        changes[attr] = (attrs1.get(attr), attrs2.get(attr))  # Show raw difference

                if changes:
                    differences.append((cls, inst_id, "MODIFIED", changes))

    return differences

# === Save Differences to CSV ===
def save_differences_to_csv(differences, output_file):
    with open(output_file, "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Class", "Instance ID", "Change Type", "Attribute", "Old Value", "New Value"])
        for cls, inst_id, change_type, details in differences:
            if change_type == "MODIFIED":
                for attr, (old, new) in details.items():
                    writer.writerow([cls, inst_id, change_type, attr, old, new])
            else:
                for attr, val in details.items():
                    writer.writerow([cls, inst_id, change_type, attr, val, ""])

# # === MAIN ===
# if __name__ == "__main__":
#     file1 = "IEEE13.xml"
#     file2 = "IEEE_13_with_Xfmr_CPOB.xml"
#     output_csv = "cim_differences.csv"

#     print("Parsing files...")
#     data1 = parse_cim(file1)
#     data2 = parse_cim(file2)

#     print("Comparing files...")
#     diffs = compare_cim_files(data1, data2)

#     print("\n=== Differences ===")
#     for cls, inst_id, change_type, details in diffs:
#         print(f"\nClass: {cls}")
#         print(f"Instance ID: {inst_id}")
#         print(f"Change Type: {change_type}")
#         if change_type == "MODIFIED":
#             for attr, (old, new) in details.items():
#                 print(f"  - {attr}: {old} → {new}")
#         else:
#             for attr, val in details.items():
#                 print(f"  - {attr}: {val}")

#     print("\nSaving differences to CSV...")
#     save_differences_to_csv(diffs, output_csv)

#     added = sum(1 for d in diffs if d[2] == "ADDED")
#     removed = sum(1 for d in diffs if d[2] == "REMOVED")
#     modified = sum(1 for d in diffs if d[2] == "MODIFIED")

#     print(f"\n=== Summary ===")
#     print(f"Added instances: {added}")
#     print(f"Removed instances: {removed}")
#     print(f"Modified instances: {modified}")
#     print(f"CSV saved as: {output_csv}")


In the given example, Object Builder is adding a power transformer using catalog parser. The CIMparator is able to compare the original network as well as the modified network.